# Parameter Calibration and Model-Assumption Checks

This notebook examines how the simulator's fill-rate and adverse-selection assumptions can be inspected from path-level data. The records here are simulated representative paths; the same workflow could be applied to permitted trade-and-quote data.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

interval_path = ROOT / "outputs" / "data" / "representative_toxic_fixed_intervals.csv"
trade_path = ROOT / "outputs" / "data" / "representative_toxic_fixed_trades.csv"
intervals = pd.read_csv(interval_path)
trades = pd.read_csv(trade_path)

intervals.head(), trades.head()

## Fill probability versus quote distance

The simulator assumes that fill intensity decreases as quote distance widens. A first calibration check is therefore to bin quote distances and compare observed fill rates across bins.

In [ ]:
bid = intervals[["bid_distance", "bid_fill_quantity"]].rename(columns={"bid_distance": "distance", "bid_fill_quantity": "fill_quantity"})
bid["side"] = "bid"
ask = intervals[["ask_distance", "ask_fill_quantity"]].rename(columns={"ask_distance": "distance", "ask_fill_quantity": "fill_quantity"})
ask["side"] = "ask"
fills = pd.concat([bid, ask], ignore_index=True)
fills["filled"] = fills["fill_quantity"] > 0
fills["distance_bin"] = pd.qcut(fills["distance"], q=6, duplicates="drop")
fill_curve = fills.groupby(["side", "distance_bin"], observed=True).agg(
    mean_distance=("distance", "mean"),
    fill_probability=("filled", "mean"),
    mean_quantity=("fill_quantity", "mean"),
    observations=("filled", "size"),
).reset_index()
display(fill_curve)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for side, group in fill_curve.groupby("side"):
    ax.plot(group["mean_distance"], group["fill_probability"], marker="o", label=side)
ax.set_xlabel("Mean quote distance from mid")
ax.set_ylabel("Observed fill probability per interval")
ax.set_title("Fill probability versus quote distance")
ax.legend()
plt.show()

## Short-horizon markout after fills

Post-fill markout is used as a toxicity diagnostic. Negative markouts mean that fills tend to be followed by price movement against the market maker.

In [ ]:
markout_by_side = trades.groupby("side").agg(
    trades=("one_step_markout", "size"),
    mean_one_step_markout=("one_step_markout", "mean"),
    median_one_step_markout=("one_step_markout", "median"),
    toxic_fraction=("is_toxic", "mean"),
    mean_spread_capture=("immediate_spread_capture", "mean"),
).reset_index()
display(markout_by_side)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(markout_by_side["side"], markout_by_side["mean_one_step_markout"])
ax.axhline(0.0, linewidth=1)
ax.set_ylabel("Mean one-step markout")
ax.set_title("Average post-fill markout by side")
plt.show()

## Extension to real data

For trade-and-quote data, the same structure would require timestamp alignment, quote filtering, side classification, queue assumptions, and a consistent markout horizon. The main research question would be whether the simulator's assumed fill-distance and markout relationships are directionally consistent with observed market behaviour.